# Demostración: Detección de Anomalías en un Proceso Electoral Simulado

> **Descargo ético obligatorio:**
> *Todos los datos de este proyecto son simulados. Los nombres de país, departamentos, municipios y candidatos son ficticios. Los resultados no representan ni hacen afirmaciones sobre ningún proceso electoral real. Una alerta generada por los detectores indica una inconsistencia que merece revisión, no una prueba de fraude.*

## 1. Cargar el dataset simulado y las dependencias del proyecto

In [ ]:
import sys
from pathlib import Path
# Asegurar que la raíz del proyecto esté en sys.path
sys.path.insert(0, str(Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()))

import pandas as pd
from IPython.display import Image, display
from src.config import DATA_RAW_CSV, DATA_PROCESSED_CSV, OUTPUTS_FIG_DIR, OUTPUTS_TAB_DIR, SEED
from src.generate_dataset import generar_dataset
from src.features import agregar_features
from src.rules import aplicar_reglas
from src.isolation_forest import aplicar_isolation_forest
from src.evaluate import ejecutar_evaluacion

# Cargar o regenerar dataset
df_raw = pd.read_csv(DATA_RAW_CSV) if DATA_RAW_CSV.exists() else generar_dataset(seed=SEED)
print(f"Total de mesas cargadas: {len(df_raw)}")
df_raw.head()

## 2. Aplicar Ingeniería de Características (Features) y Reglas Simples

In [ ]:
df_feat = agregar_features(df_raw)
df_reglas = aplicar_reglas(df_feat)
print(f"Mesas alertadas por reglas simples: {df_reglas['alerta_reglas'].sum()}")
df_reglas['severidad_reglas'].value_counts()

## 3. Aplicar Detección no supervisada con Isolation Forest

In [ ]:
df_proc, modelo_if, scaler = aplicar_isolation_forest(df_reglas, seed=SEED)
print(f"Mesas alertadas por Isolation Forest (top 4% outliers): {df_proc['alerta_if'].sum()}")
df_proc[['mesa_id', 'participacion', 'pct_max_candidato', 'score_if', 'alerta_if', 'ranking_if']].head()

## 4. Evaluación contra Ground Truth y Métricas de Clasificación

In [ ]:
df_final, tablas = ejecutar_evaluacion(df_proc)
display(tablas['metricas'])
display(tablas['recall_tipo'])

## 5. Visualizaciones Oficiales del Proceso

In [ ]:
figuras = [
    "fig1_distribucion_participacion.png",
    "fig2_huella_electoral.png",
    "fig3_score_isolation_forest.png",
    "fig4_recall_por_tipo.png",
    "fig5_riesgo_por_municipio.png"
]
for f in figuras:
    print(f"=== {f} ===")
    display(Image(filename=str(OUTPUTS_FIG_DIR / f)))


## 6. Top 5 Mesas más Sospechosas y Explicación en Detalle

In [ ]:
top_5 = tablas['top_15'].head(5)
display(top_5)

mesa_ejemplo = top_5.iloc[0]
print("\n--- EXPLICACIÓN DE LA MESA #1 PRIORIZADA PARA AUDITORÍA ---")
print(f"Mesa ID: {mesa_ejemplo['mesa_id']} en {mesa_ejemplo['municipio']} ({mesa_ejemplo['departamento']})")
print(f"Votantes habilitados: {mesa_ejemplo['votantes_habilitados']} | Total reportado: {mesa_ejemplo['total_votos_reportado']}")
print(f"Participación: {mesa_ejemplo['participacion'] * 100:.2f} % (Sobre-participación física imposible)")
print(f"Nivel de riesgo: {mesa_ejemplo['nivel_riesgo']} (Severidad reglas: {mesa_ejemplo['severidad_reglas']}, Score IF: {mesa_ejemplo['score_if']:.4f})")
print(f"Tipo real inyectado (Ground Truth): {mesa_ejemplo['tipo_anomalia']}")
print("Acción recomendada: Apertura inmediata del sobre electoral para recuento manual papeleta por papeleta.")